# 08 — Benchmark de escala: a mesma NeuMF sobre uma base **sintética rotulada**

Este notebook roda o **pipeline de recomendação real** (`src/`) sobre uma base
**100 % sintética e reproduzível**, para demonstrar o comportamento do sistema
**em escala**, onde existe sinal colaborativo de verdade.

> **Honestidade em primeiro lugar.** A base aqui é **sintética** (gerada por um
> processo documentado, semente fixa) e **não representa dados reais de nenhuma
> empresa**. O que se demonstra é a **arquitetura e o método** — não "os dados".

## 1. Por que uma base sintética?

A base pública real (*sc_base_01*) é **pequena e esparsa** (~800 clientes,
~12 itens/cliente, densidade 0,66 %). Nessa escala, o *baseline* de
*mais-populares* é o sinal mais forte e **nenhum** modelo neural o supera de
forma estável — resultado honesto reportado no notebook `07`.

Um recomendador "nível produção" só **demonstra personalização** quando existe
sinal colaborativo recuperável. Como dados reais de clientes são protegidos por
privacidade (o que se publica em portfólio é sempre anonimizado/sintético),
construímos um **benchmark sintético** com estrutura latente real e **o mesmo
código do pipeline**. Assim provamos que:

1. o método **recupera preferência latente** quando ela existe;
2. o ganho sobre o *baseline* de popularidade é **grande e mensurável**;
3. existe um **teto teórico** (recuperável) — e medimos quanto dele capturamos.

## 2. Configuração do ambiente

In [1]:
import sys
from pathlib import Path

# Localiza a raiz do projeto (onde está o pyproject.toml) para importar `src.*`.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

import json

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

from src.config.paths import REPORTS_DIR
from src.config.model_utils import (
    RANDOM_SEED,
    TOP_K,
    RankingCheckpoint,
    build_item_catalog,
    build_training_pairs,
    compute_popularity_prior,
    create_ranking_model,
    evaluate_from_scores,
    evaluate_hybrid,
    evaluate_most_popular,
    evaluate_random_baseline,
    evaluate_ranking,
    select_alpha,
    to_model_inputs,
    train_ranking_model,
    train_validation_test_split,
)
from src.config.synthetic_data import dataset_summary, generate_synthetic_dataset

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
print("Projeto:", ROOT)
print("TensorFlow:", tf.__version__)

Projeto: C:\Users\flavi\Documents\GitHub\Sistema_Recomendacao_Arquitetura-MLOps
TensorFlow: 2.19.0


## 3. A base sintética (reprodutível)

O gerador (`src/config/synthetic_data.py`) cria:

- **fatores latentes** de usuário e item (produto interno define a afinidade);
- **popularidade em cauda longa** (log-normal) → um *baseline* de *mais vendidos* não-trivial;
- amostragem `p_ui ∝ s_i · exp(preference_strength · a_ui)`, com `preference_strength`
  **calibrado** (`= 20`) para a afinidade latente dominar a popularidade.

Escala do benchmark: **3.000 clientes × ~1.000 produtos**, ~22 itens/cliente.

In [2]:
N_USERS = 3000
N_ITEMS = 1000
AVG_INTERACTIONS = 22

df, fatores = generate_synthetic_dataset(
    n_users=N_USERS, n_items=N_ITEMS, avg_interactions=AVG_INTERACTIONS,
    return_factors=True,  # mantém os fatores verdadeiros para o teto "oracle"
)
resumo = dataset_summary(df)
for chave, valor in resumo.items():
    print(f"{chave:<26}: {valor}")

interacoes                : 66020
clientes                  : 3000
produtos                  : 990
categorias                : 8
subcategorias             : 40
itens_por_cliente_media   : 22.006666666666668
itens_por_cliente_mediana : 22.0
densidade                 : 0.02222895622895623


## 4. Split *leave-last-out*, catálogo e pares de treino

Mesmo protocolo do pipeline de produção: último item → **teste**, penúltimo →
**validação**, o restante → **treino** (com negativos amostrados).

In [3]:
colunas = ["Customer Name", "Product ID", "Product Name", "Category", "Sub-Category", "Order Date"]
df = df[colunas].copy()

encoders = {
    "customer": LabelEncoder().fit(df["Customer Name"]),
    "product": LabelEncoder().fit(df["Product ID"]),
    "category": LabelEncoder().fit(df["Category"]),
    "subcategory": LabelEncoder().fit(df["Sub-Category"]),
}
num_clientes = len(encoders["customer"].classes_)
num_produtos = len(encoders["product"].classes_)
num_categorias = len(encoders["category"].classes_)
num_subcategorias = len(encoders["subcategory"].classes_)

catalogo = build_item_catalog(df, encoders["product"], encoders["category"], encoders["subcategory"])
train_pairs, val_pairs, test_pairs = train_validation_test_split(df, encoders["customer"], encoders["product"])

N_NEGATIVES = 8
train_customers, train_products, train_labels = build_training_pairs(
    train_pairs, num_produtos, n_negatives=N_NEGATIVES, seed=RANDOM_SEED
)
val_customers, val_products, val_labels = build_training_pairs(
    val_pairs, num_produtos, n_negatives=4, seed=RANDOM_SEED, exclude_pairs=train_pairs
)

print(f"Dimensões: {num_clientes} clientes | {num_produtos} produtos | "
      f"{num_categorias} categorias | {num_subcategorias} sub-categorias")
print(f"Pares -> treino: {len(train_pairs)} | validação: {len(val_pairs)} | teste: {len(test_pairs)}")
print(f"Treino: {len(train_labels)} exemplos ({int(train_labels.sum())} positivos / "
      f"{int((train_labels == 0).sum())} negativos)")

Dimensões: 3000 clientes | 990 produtos | 8 categorias | 40 sub-categorias
Pares -> treino: 60020 | validação: 3000 | teste: 3000
Treino: 540180 exemplos (60020 positivos / 480160 negativos)


## 5. Treino da NeuMF (early stopping por NDCG@7)

Arquitetura **idêntica à de produção** (GMF + MLP, embeddings, sigmoide). A
**única** diferença são os hiperparâmetros de regularização:

> Na base real (minúscula) usa-se `L2=1e-3` + `dropout=0,3` **por necessidade**
> (evitar decoração). Em escala, essa mesma regularização **sufoca** a
> personalização — então, aqui, a validação seleciona um modelo praticamente
> **sem regularização**, que de fato aprende a preferência latente.

In [4]:
EMBEDDING_DIM = 32
L2_STRENGTH = 0.0
DROPOUT = 0.0
EPOCHS = 40
BATCH_SIZE = 512
PATIENCE = 5

modelo = create_ranking_model(
    num_clientes, num_produtos, num_categorias, num_subcategorias,
    embedding_dim=EMBEDDING_DIM, l2_strength=L2_STRENGTH,
    dropout_rate=DROPOUT, seed=RANDOM_SEED,
)
ranking_checkpoint = RankingCheckpoint(val_pairs, train_pairs, catalogo, k=TOP_K, patience=PATIENCE, verbose=1)
callbacks = [
    ranking_checkpoint,
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
]
history = train_ranking_model(
    modelo, train_customers, train_products, catalogo[1][train_products], catalogo[2][train_products], train_labels,
    validation_data=(to_model_inputs(val_customers, val_products, catalogo), val_labels),
    epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=2,
)
print(f"Treino encerrado em {len(history.history['loss'])} épocas | "
      f"melhor NDCG@{TOP_K} de validação: {ranking_checkpoint.best:.5f} (época {ranking_checkpoint.best_epoch})")

Epoch 1/40


    [ranking] época 1: val_ndcg@7=0.04603 (melhor=0.04603 na época 1)
1056/1056 - 7s - 7ms/step - auc: 0.7606 - loss: 0.3052 - val_auc: 0.8424 - val_loss: 0.4012 - val_ndcg@7: 0.0460 - learning_rate: 0.0010


Epoch 2/40


    [ranking] época 2: val_ndcg@7=0.07384 (melhor=0.07384 na época 2)
1056/1056 - 5s - 5ms/step - auc: 0.9016 - loss: 0.2197 - val_auc: 0.8973 - val_loss: 0.3331 - val_ndcg@7: 0.0738 - learning_rate: 0.0010


Epoch 3/40


    [ranking] época 3: val_ndcg@7=0.09240 (melhor=0.09240 na época 3)
1056/1056 - 5s - 5ms/step - auc: 0.9284 - loss: 0.1927 - val_auc: 0.9112 - val_loss: 0.3155 - val_ndcg@7: 0.0924 - learning_rate: 0.0010


... (17 outputs left out)

1056/1056 - 5s - 5ms/step - auc: 0.9774 - loss: 0.1139 - val_auc: 0.9233 - val_loss: 0.3290 - val_ndcg@7: 0.1009 - learning_rate: 5.0000e-04


Epoch 12/40


    [ranking] época 12: val_ndcg@7=0.10065 (melhor=0.10769 na época 7)
1056/1056 - 6s - 5ms/step - auc: 0.9796 - loss: 0.1087 - val_auc: 0.9241 - val_loss: 0.3352 - val_ndcg@7: 0.1006 - learning_rate: 2.5000e-04


Treino encerrado em 12 épocas | melhor NDCG@7 de validação: 0.10769 (época 7)


## 6. Baselines e o teto *oracle*

Além do **mais-populares** e do **aleatório** (esperança analítica exata),
medimos um teto **"oracle"**: ranquear cada cliente pela **afinidade latente
verdadeira** (o sinal que gerou os dados). É o máximo recuperável sob o mesmo
protocolo — a régua que diz quanto do potencial o modelo captura.

In [5]:
popularity = compute_popularity_prior(train_pairs, num_produtos)
alpha, historico_alpha = select_alpha(modelo, val_pairs, train_pairs, catalogo, popularity, k=TOP_K)
print(f"alpha escolhido na validação: {alpha}")

metricas_hybrid = evaluate_hybrid(modelo, test_pairs, train_pairs, catalogo, popularity, alpha, k=TOP_K)
metricas_modelo = evaluate_ranking(modelo, test_pairs, train_pairs, catalogo, k=TOP_K)
metricas_populares = evaluate_most_popular(test_pairs, train_pairs, num_produtos, k=TOP_K)
metricas_aleatorio = evaluate_random_baseline(test_pairs, train_pairs, num_produtos, k=TOP_K)

# Teto "oracle": ranqueia pela afinidade latente verdadeira.
u_src = np.array([int(nome.split()[1]) for nome in encoders["customer"].classes_], dtype="int64")
p_src = np.array([int(pid.split("-")[1]) for pid in encoders["product"].classes_], dtype="int64")
usuarios = fatores["user_latent"][u_src]
itens = fatores["item_latent"][p_src]
customers = np.asarray(sorted(test_pairs["c"].unique()), dtype="int64")
metricas_oracle = evaluate_from_scores(usuarios[customers] @ itens.T, train_pairs, test_pairs, k=TOP_K)

# Seleção do modelo final POR VALIDAÇÃO (NeuMF puro x híbrido).
val_neumf = float(ranking_checkpoint.best)
val_hibrido = float(historico_alpha[alpha])
if val_neumf >= val_hibrido:
    nome_final, metricas_final = "NeuMF puro", metricas_modelo
else:
    nome_final, metricas_final = f"Híbrido popularity-first (α={alpha})", metricas_hybrid
print("Modelo final (escolhido por validação):", nome_final)

alpha escolhido na validação: 1.0


Modelo final (escolhido por validação): NeuMF puro


## 7. Resultados no teste

Métricas *leave-last-out*, **catálogo completo**, `k = 7`.

In [6]:
colunas_metricas = ["ndcg@7", "map@7", "hit_rate@7", "precision@7", "recall@7", "f1@7"]
politicas = {
    "Aleatório (esperança)": metricas_aleatorio,
    "Mais-populares": metricas_populares,
    f"Híbrido popularity-first (α={alpha})": metricas_hybrid,
    "NeuMF puro": metricas_modelo,
    "Oracle (afinidade latente)": metricas_oracle,
}
tabela = pd.DataFrame(
    {nome: {c: m.get(c, np.nan) for c in colunas_metricas} for nome, m in politicas.items()}
).T
tabela.index.name = "Política"

ndcg_final = metricas_final["ndcg@7"]
ndcg_pop = metricas_populares["ndcg@7"]
ndcg_oracle = metricas_oracle["ndcg@7"]
print(f"Ganho de NDCG@7 do modelo final sobre 'Mais-populares': {ndcg_final / ndcg_pop - 1:+.1%}")
print(f"Fração do teto 'oracle' capturada pelo modelo final: {ndcg_final / ndcg_oracle:.1%}")
display(tabela.round(4))

Ganho de NDCG@7 do modelo final sobre 'Mais-populares': +431.7%
Fração do teto 'oracle' capturada pelo modelo final: 81.5%


,ndcg@7,map@7,hit_rate@7,precision@7,recall@7,f1@7
Política,,,,,,
Aleatório (esperança),NaN,NaN,0.0072,0.0010,0.0072,NaN
Mais-populares,0.0215,0.0160,0.0390,0.0056,0.0390,0.0098
Híbrido popularity-first (α=1.0),0.1085,0.0835,0.1877,0.0268,0.1877,0.0469
NeuMF puro,0.1141,0.0882,0.1967,0.0281,0.1967,0.0492
Oracle (afinidade latente),0.1400,0.1060,0.2483,0.0355,0.2483,0.0621


In [7]:
# Figura: NDCG@7 por política (salva em report/figuras/).
FIG_DIR = REPORTS_DIR / "figuras"
FIG_DIR.mkdir(parents=True, exist_ok=True)

rotulos = ["Mais-populares", f"Híbrido (α={alpha})", "NeuMF (final)", "Oracle (afinidade)"]
valores = [ndcg_pop, metricas_hybrid["ndcg@7"], ndcg_final, ndcg_oracle]

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.bar(rotulos, valores, color=["#9e9e9e", "#5c6bc0", "#1e88e5", "#43a047"])
ax.bar_label(barras, fmt="%.4f", padding=2)
ax.set_ylabel("NDCG@7 (teste)")
ax.set_title("Benchmark de escala — NDCG@7 por política")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
caminho_fig = FIG_DIR / "benchmark_escala_ndcg.png"
fig.savefig(caminho_fig, dpi=130)
plt.close(fig)
print("Figura salva em:", caminho_fig)

Figura salva em: C:\Users\flavi\Documents\GitHub\Sistema_Recomendacao_Arquitetura-MLOps\report\figuras\benchmark_escala_ndcg.png


In [8]:
# Persistência das métricas (artefato reproduzível do benchmark).
saida = {
    "base": "sintetica (reprodutivel, seed=42)",
    "resumo_base": resumo,
    "alpha_hibrido": float(alpha),
    "modelo_final": nome_final,
    "val_ndcg_final": float(val_neumf if nome_final == "NeuMF puro" else val_hibrido),
    "val_ndcg_neumf": float(val_neumf),
    "metricas_teste": {
        "final": metricas_final,
        "hibrido": metricas_hybrid,
        "neumf_puro": metricas_modelo,
        "mais_populares": metricas_populares,
        "oracle_afinidade": metricas_oracle,
        "aleatorio": metricas_aleatorio,
    },
    "ganho_ndcg_sobre_populares": float(ndcg_final / ndcg_pop - 1),
}
caminho_json = REPORTS_DIR / "benchmark_sintetico_metricas.json"
caminho_json.write_text(json.dumps(saida, indent=2, ensure_ascii=False), encoding="utf-8")
print("Métricas salvas em:", caminho_json)

Métricas salvas em: C:\Users\flavi\Documents\GitHub\Sistema_Recomendacao_Arquitetura-MLOps\report\benchmark_sintetico_metricas.json


## 8. Leitura sênior

- **O método funciona em escala.** Com sinal colaborativo real, a NeuMF
  **supera o baseline de popularidade por ~4×** e captura a **maior parte do teto
  recuperável** — ou seja, **não é** um recomendador de *mais vendidos*.
- **A limitação da base real é de dados, não de código.** O mesmo pipeline, na
  base pública pequena, é honestamente dominado pela popularidade (notebook `07`).
- **Honestidade explícita.** A base deste notebook é **sintética e rotulada como
  tal**; os números são **reais** (execução deste notebook), mas os *dados* não
  são de nenhuma empresa. O que se prova é **a arquitetura e o método**.

> Artefato reproduzível: `report/benchmark_sintetico_metricas.json`
> · Script equivalente: `python -m src.scripts.benchmark_sintetico`.